## Load Data

In [1]:
import requests #to communicate with API
from pathlib import Path 

#send get request and convert response to json (List of files)
files = requests.get("https://api.figshare.com/v2/articles/21316608").json()["files"]
f = files[0]

print(f["name"], round(f["size"] / 1e6), "MB")   # check the size before downloading

20180108T184332_dmi_prep.nc 146 MB


In [2]:
out = Path("data"); out.mkdir(exist_ok=True) #Make directory for loaded data (if not exists)

path = out / f["name"] #here/2018.nc eg

with requests.get(f["download_url"], stream=True) as r:
    r.raise_for_status()
    with open(path, "wb") as fh:
        for chunk in r.iter_content(1 << 20):
            fh.write(chunk)

In [ ]:
import xarray as xr

ds = xr.open_dataset(path) #Representation for NetCDF (.nc) file

print(ds)

<xarray.Dataset> Size: 530MB
Dimensions:               (sar_lines: 5353, sar_samples: 5180,
                           sar_sample_2dgrid_points: 23,
                           sar_line_2dgrid_points: 21, 2km_grid_lines: 214,
                           2km_grid_samples: 207)
Dimensions without coordinates: sar_lines, sar_samples,
                                sar_sample_2dgrid_points,
                                sar_line_2dgrid_points, 2km_grid_lines,
                                2km_grid_samples
Data variables: (12/29)
    SIC                   (sar_lines, sar_samples) uint8 28MB ...
    SOD                   (sar_lines, sar_samples) uint8 28MB ...
    FLOE                  (sar_lines, sar_samples) uint8 28MB ...
    sar_grid2d_latitude   (sar_sample_2dgrid_points, sar_line_2dgrid_points) float64 4kB ...
    sar_grid2d_longitude  (sar_sample_2dgrid_points, sar_line_2dgrid_points) float64 4kB ...
    nersc_sar_primary     (sar_lines, sar_samples) float32 111MB ...
    ...      

In [7]:
for name in ds.data_vars:
    print(name)

SIC
SOD
FLOE
sar_grid2d_latitude
sar_grid2d_longitude
nersc_sar_primary
nersc_sar_secondary
sar_incidenceangle
distance_map
btemp_6_9h
btemp_6_9v
btemp_7_3h
btemp_7_3v
btemp_10_7h
btemp_10_7v
btemp_18_7h
btemp_18_7v
btemp_23_8h
btemp_23_8v
btemp_36_5h
btemp_36_5v
btemp_89_0h
btemp_89_0v
u10m_rotated
v10m_rotated
t2m
skt
tcwv
tclw


SIC = Sea Ice Concentration, so this is the target to predict
We want to evaluate based off of satellite imagery, so the SAR variables is what we should start with
We still need to split SIC into train and test data, as the NN will need to find an error on each batch and update weights through backpropagation to reduce this
Eventually we can add more variables to see how the CNN improves its predictions